<a href="https://colab.research.google.com/github/leonardo2004/Calculo-Numerico/blob/main/Atividade_01/Atividade01_CalcNumerico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade 01 - Cálculo Numérico

## Integrantes e Divisão de Tarefas
* **Leonardo Tomasela Leal (RA: 170291)**: Implementação das funções de erro, lógica de truncamento e desenvolvimento da animação Manim.
* **Giovanna Maria de Siqueira Junqueira (RA: 185218)**: Análise de sensibilidade física, elaboração dos gráficos Matplotlib e redação das conclusões técnicas.

In [ ]:
# Instalação do Manim e dependências do sistema
!sudo apt update -y && sudo apt install -y libcairo2-dev libpango1.0-dev ffmpeg freeglut3-dev pkg-config libffi-dev qtbase5-dev libqt5svg5-dev
!pip install manim

import math
import numpy as np
import pandas as pd
import os
import shutil
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from manim import *

# Configurações estéticas dos gráficos
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (12, 8), 'font.size': 12})

# --- Funções de Erro (Padronizadas PEP8) ---

def calcular_erro_absoluto(valor_referencia, valor_aproximado):
    """Calcula a diferença absoluta entre o valor real e a aproximação."""
    return abs(valor_referencia - valor_aproximado)

def calcular_erro_relativo(valor_referencia, valor_aproximado):
    """Calcula a razão do erro absoluto pelo valor de referência."""
    if valor_referencia == 0:
        return 0.0
    return calcular_erro_absoluto(valor_referencia, valor_aproximado) / abs(valor_referencia)

def calcular_erro_percentual(valor_referencia, valor_aproximado):
    """Converte o erro relativo em porcentagem."""
    return 100 * calcular_erro_relativo(valor_referencia, valor_aproximado)

# Versões vetorizadas para performance com NumPy
vec_erro_absoluto = np.vectorize(calcular_erro_absoluto)
vec_erro_relativo = np.vectorize(calcular_erro_relativo)
vec_erro_percentual = np.vectorize(calcular_erro_percentual)

# Parte A


In [ ]:
def truncar(x, n):
    return math.trunc(x * (10 ** n)) / (10 ** n)

valores = {"x1": 3.14159265, "x2": 98.76543, "x3": 0.00498765}
resultados = []

for nome, x in valores.items():
    for n in range(5):
        x_arr = round(x, n)
        ea_arr = abs(x - x_arr)
        resultados.append({"Valor": nome, "n": n, "Método": "Arredondamento", "Ea": ea_arr})

        x_tr = truncar(x, n)
        ea_tr = abs(x - x_tr)
        resultados.append({"Valor": nome, "n": n, "Método": "Truncamento", "Ea": ea_tr})

df = pd.DataFrame(resultados)

# Visualização em Subplots Lado a Lado
plt.style.use('seaborn-v0_8-darkgrid')
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

for i, nome in enumerate(valores.keys()):
    df_val = df[df["Valor"] == nome]
    arr = df_val[df_val["Método"] == "Arredondamento"]
    tr = df_val[df_val["Método"] == "Truncamento"]

    axs[i].plot(arr["n"], arr["Ea"], marker='o', label='Arredondamento', lw=2)
    axs[i].plot(tr["n"], tr["Ea"], marker='s', ls='--', label='Truncamento', lw=2)
    axs[i].set_yscale('log')
    axs[i].set_title(f'Erro Absoluto - {nome}', fontweight='bold')
    axs[i].set_xlabel('n (casas)')
    axs[i].legend()

plt.tight_layout()
plt.show()

# Parte B

## B.1

In [ ]:
temp = np.array([121.7, 120.4, 122.1, 119.8, 121.2, 120.9, 122.4, 121.0, 120.2, 121.5])

pressao = {
    "Dados originais" : temp,
    "Dados arredondados" : np.round(temp,0),
    "Dados truncados" : np.trunc(temp),
}

del temp

pressao_mean = {}
dictabs_error = {}
dictrel_error = {}
dictper_error = {}
for item, lista in pressao.items():
  pressao_mean[item] = lista.mean()
  print(f"{item} : {lista}\n"
  f"média: {pressao_mean[item]}\n"
  f"min: {lista.min()}\n"
  f"max: {lista.max()}\n"
  f"amplitude {lista.max()-lista.min():e}\n"
  f"desvio {lista.std():e}\n")
  dictabs_error[item] = abs(pressao_mean["Dados originais"]-pressao_mean[item])
  dictrel_error[item] = dictabs_error[item]/pressao_mean["Dados originais"]
  dictper_error[item] = 100*dictrel_error[item]
  print(f"Erro absoluto = {dictabs_error[item]:e}\n"
  f"Erro relativo = {dictrel_error[item]:e}\n"
  f"Erro percentual = {dictper_error[item]:e}\n")



## B.2

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
fig = plt.figure(figsize=(18, 10))
grid = fig.add_gridspec(2, 2)

# 1. Gráfico de Linha - Comparação
ax1 = fig.add_subplot(grid[0, 0])
x = np.arange(10)
ax1.plot(x, pressao['Dados originais'], label='Original', marker='o')
ax1.plot(x, pressao['Dados arredondados'], label='Arred.', ls='--', marker='s')
ax1.plot(x, pressao['Dados truncados'], label='Trunc.', ls=':', marker='^')
ax1.set_title('Mediciones de Presión', fontweight='bold')
ax1.legend()

# 2. Comparação de Erros (Absoluto e Relativo)
ax2 = fig.add_subplot(grid[0, 1])
x_pos = np.arange(len(dictabs_error))
width = 0.35
ax2.bar(x_pos - width/2, dictabs_error.values(), width, label='Ea', alpha=0.8)
ax2.bar(x_pos + width/2, dictrel_error.values(), width, label='Er', alpha=0.8)
ax2.set_xticks(x_pos, dictabs_error.keys())
ax2.set_title('Ea vs Er por Método', fontweight='bold')
ax2.legend()

# 3. Distribuição (Histogramas Agrupados)
ax3 = fig.add_subplot(grid[1, :])
ax3.hist([pressao['Dados originais'], pressao['Dados arredondados'], pressao['Dados truncados']],
         label=['Original', 'Arred.', 'Trunc.'], bins=5)
ax3.set_title('Distribuição das Pressões', fontweight='bold')
ax3.legend()

plt.tight_layout()
plt.show()

# Parte C


## C.1


In [ ]:
def truncar_valor(valor, casas_decimais):
    fator = 10 ** casas_decimais
    return math.trunc(valor * fator) / fator

def calcular_vazao_poiseuille(raio_mm, viscosidade, delta_p, comprimento=0.20):
    raio_metros = raio_mm * 1e-3
    numerador = np.pi * (raio_metros ** 4) * delta_p
    denominador = 8 * viscosidade * comprimento
    return numerador / denominador

raio_ref = 0.80
viscosidade_ref = 3.5e-3
pressao_ref = 1200.0
comprimento_ref = 0.20
vazao_real = calcular_vazao_poiseuille(raio_ref, viscosidade_ref, pressao_ref, comprimento_ref)

cenarios = [
    {"Cenário": "Referência (Exato)", "r": raio_ref, "mu": viscosidade_ref, "p": pressao_ref},
    {"Cenário": "Raio Arredondado (1 casa)", "r": round(raio_ref, 1), "mu": viscosidade_ref, "p": pressao_ref},
    {"Cenário": "Visc. Truncada (3 casas)", "r": raio_ref, "mu": truncar_valor(viscosidade_ref, 3), "p": pressao_ref},
    {"Cenário": "Pressão Arred. (Centenas)", "r": raio_ref, "mu": viscosidade_ref, "p": round(pressao_ref, -2)},
    {"Cenário": "Simultâneo (Tudo Arred.)", "r": round(raio_ref, 1), "mu": round(viscosidade_ref, 3), "p": round(pressao_ref, -2)}
]

lista_resultados = []
for c in cenarios:
    v_aprox = calcular_vazao_poiseuille(c['r'], c['mu'], c['p'])
    lista_resultados.append({
        "Cenário": c['Cenário'],
        "Vazão [m³/s]": v_aprox,
        "Erro Absoluto [m³/s]": abs(vazao_real - v_aprox),
        "Erro Percentual [%]": (abs(vazao_real - v_aprox) / vazao_real) * 100 if vazao_real != 0 else 0
    })

df_c1 = pd.DataFrame(lista_resultados)
display(df_c1)

## C.2


In [ ]:
def truncar(x, n):
    return math.trunc(x * (10 ** n)) / (10 ** n)

# Dados fornecidos
L_val = 0.20
mu_orig = 3.5e-3
dp_orig = 1200.0

# Função da vazão Q
def calcula_Q(r_mm, mu=mu_orig, dp=dp_orig, L=L_val):
    r_m = r_mm * 1e-3
    return (np.pi * (r_m ** 4) * dp) / (8 * mu * L)


r_valores = np.linspace(0.70, 0.90, 201)

resultados_r = []

for r in r_valores:
    r_ref = r
    r_arr = round(r, 1)
    r_tru = truncar(r, 1)

    Q_ref = calcula_Q(r_ref)
    Q_arr = calcula_Q(r_arr)
    Q_tru = calcula_Q(r_tru)

    # Erros do Arredondamento
    Ea_arr = abs(Q_ref - Q_arr)
    Er_arr = Ea_arr / Q_ref
    Ep_arr = Er_arr * 100

    # Erros do Truncamento
    Ea_tru = abs(Q_ref - Q_tru)
    Er_tru = Ea_tru / Q_ref
    Ep_tru = Er_tru * 100

    resultados_r.append({
        'r_mm': r,
        'Q_ref': Q_ref,
        'Q_arr': Q_arr, 'Ea_arr': Ea_arr, 'Er_arr': Er_arr, 'Ep_arr': Ep_arr,
        'Q_tru': Q_tru, 'Ea_tru': Ea_tru, 'Er_tru': Er_tru, 'Ep_tru': Ep_tru
    })

df_r = pd.DataFrame(resultados_r)

# Criação dos Gráficos
fig, axs = plt.subplots(2, 2, figsize=(14, 10))

# 1. Q em função de r
axs[0, 0].plot(df_r['r_mm'], df_r['Q_ref'], label='Q Referência', color='black', lw=2)
axs[0, 0].plot(df_r['r_mm'], df_r['Q_arr'], label='Q Arredondado (1 casa)', linestyle='--', color='blue')
axs[0, 0].plot(df_r['r_mm'], df_r['Q_tru'], label='Q Truncado (1 casa)', linestyle=':', color='red')
axs[0, 0].set_xlabel('Raio r (mm)')
axs[0, 0].set_ylabel('Vazão Q ($m^3/s$)')
axs[0, 0].set_title('1. Vazão Q em função de r')
axs[0, 0].grid(True)
axs[0, 0].legend()

# 2. Erro Absoluto de Q em função de r
axs[0, 1].plot(df_r['r_mm'], df_r['Ea_arr'], label='Arredondamento', color='blue')
axs[0, 1].plot(df_r['r_mm'], df_r['Ea_tru'], label='Truncamento', color='red', linestyle='--')
axs[0, 1].set_xlabel('Raio r (mm)')
axs[0, 1].set_ylabel('Erro Absoluto $E_a$')
axs[0, 1].set_title('2. Erro Absoluto de Q em função de r')
axs[0, 1].grid(True)
axs[0, 1].legend()

# 3. Erro Relativo de Q em função de r
axs[1, 0].plot(df_r['r_mm'], df_r['Er_arr'], label='Arredondamento', color='blue')
axs[1, 0].plot(df_r['r_mm'], df_r['Er_tru'], label='Truncamento', color='red', linestyle='--')
axs[1, 0].set_xlabel('Raio r (mm)')
axs[1, 0].set_ylabel('Erro Relativo $E_r$')
axs[1, 0].set_title('3. Erro Relativo de Q em função de r')
axs[1, 0].grid(True)
axs[1, 0].legend()

# 4. Erro Percentual de Q em função de r
axs[1, 1].plot(df_r['r_mm'], df_r['Ep_arr'], label='Arredondamento', color='blue')
axs[1, 1].plot(df_r['r_mm'], df_r['Ep_tru'], label='Truncamento', color='red', linestyle='--')
axs[1, 1].set_xlabel('Raio r (mm)')
axs[1, 1].set_ylabel('Erro Percentual $E_\%$ (%)')
axs[1, 1].set_title('4. Erro Percentual de Q em função de r')
axs[1, 1].grid(True)
axs[1, 1].legend()

plt.tight_layout()
plt.show()

# Parte D


In [ ]:
r = 50e-9 #m
L = 10e-6 #m
mu = 3.5e-3 #Pas
Delta_p = 5000 # Pa

Q_ref = calcula_Q(r, mu, Delta_p, L)
vec_calc_Q = np.vectorize(calcula_Q)
print(f"Valor de Q(r) referência: {Q_ref:.2e}")

## D.1

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Redefinindo as funções para garantir que não haja conflito com os dicionários da Parte B
def calc_abs_error(referencia, aproximacao):
    return abs(referencia - aproximacao)

def calc_rel_error(referencia, aproximacao):
    return abs(referencia - aproximacao) / abs(referencia) if referencia != 0 else 0

def calc_per_error(referencia, aproximacao):
    return 100 * calc_rel_error(referencia, aproximacao)

vec_abs_error = np.vectorize(calc_abs_error)
vec_rel_error = np.vectorize(calc_rel_error)
vec_per_error = np.vectorize(calc_per_error)

Delta_r = np.linspace(0.1e-9, 5e-9, 50)
r_minus = np.array(r - Delta_r)
r_plus = np.array(r + Delta_r)

df = pd.DataFrame()
df["Q(r_-)"] = vec_calc_Q(r_minus, mu, Delta_p, L)
df["Q(r)"] = vec_calc_Q(r, mu, Delta_p, L)
df["Q(r_+)"] = vec_calc_Q(r_plus, mu, Delta_p, L)
df["Delta_r"] = Delta_r

# Gráfico de Vazão
plt.figure(figsize=(14, 7))
cores = ['#2E86C1', '#E74C3C', '#27AE60']
marcadores = ['o', 's', '^']
for i, coluna in enumerate(['Q(r_-)', 'Q(r)', 'Q(r_+)']):
    plt.plot(df['Delta_r'] * 1e9, df[coluna], label=coluna, color=cores[i], marker=marcadores[i], markersize=6, alpha=0.8)
plt.xlabel('Δr (nm)')
plt.ylabel('Q (vazão)')
plt.title('Vazão em função da variação do raio')
plt.legend()
plt.show()

# Cálculo dos Erros usando as novas referências de função
df["Abs_Error_Q(r_+)"] = vec_abs_error(df["Q(r_+)"], df["Q(r)"])
df["Rel_Error_Q(r_+)"] = vec_rel_error(df["Q(r_+)"], df["Q(r)"])
df["Per_Error_Q(r_+)"] = vec_per_error(df["Q(r_+)"], df["Q(r)"])

# Gráfico Erro Relativo (Onde o erro ocorria)
plt.figure(figsize=(14, 7))
plt.plot(df['Delta_r'] * 1e9, df["Rel_Error_Q(r_+)"], label='Erro Relativo Q(r_+)', color='#2E86C1', marker='s')
idx_max = df["Rel_Error_Q(r_+)"].idxmax()
delta_max = df.loc[idx_max, 'Delta_r'] * 1e9
erro_rel_max = df["Rel_Error_Q(r_+)"].max()
plt.title('Erro Relativo da Vazão para r_+')
plt.show()

df["Delta_Q/Q(r)"] = (df['Q(r)'] - df['Q(r_-)']) / df["Q(r)"]
df["4*Delta_r/r"] = 4 * df["Delta_r"] / r

plt.figure(figsize=(14, 7))
plt.plot(df['Delta_r'] * 1e9, df["Delta_Q/Q(r)"], label='ΔQ/Q(r)', color='#E74C3C', marker='o')
plt.plot(df['Delta_r'] * 1e9, df["4*Delta_r/r"], label='4Δr/r', color='#2E86C1', ls='--')
plt.legend()
plt.title('Comparação: ΔQ/Q(r) vs 4Δr/r')
plt.show()

## D.2

In [ ]:
df = pd.DataFrame

r_32 = np.float32(r)
r_64 = np.float64(r)

print(f"R⁴                 - Float 32: {r_32**4} || Float 64: {r_64**4}\n"+
      f"Q                  - Float 32: {calcula_Q(r_32,mu,Delta_p,L)} || Float 64: {calcula_Q(r_64,mu,Delta_p,L)}\n"+
      f"Diferença absoluta - R⁴      : {abs(r_32-r_64)}  || Q       : {abs(calcula_Q(r_32,mu,Delta_p,L) - calcula_Q(r_64,mu,Delta_p,L))}\n"+
      f"np.finfo().eps     - float32 : {np.finfo(np.float32).eps} || float64 : {np.finfo(np.float64).eps}")


def delta_1(Q,r,Delta_r):
  try:
    return (Q(r+Delta_r) - Q(r))/Q(r)
  except ZeroDivisionError:
    print("Q(r) é zero, portanto erro de divisão por zero")

vec_delta_1 = np.vectorize(delta_1)

def delta_2(r,Delta_r):
  try:
    return (1 + Delta_r/r)**4 - 1
  except ZeroDivisionError:
    print("r é zero, portanto erro de divisão por zero")

vec_delta_2 = np.vectorize(delta_2)


resultados = pd.DataFrame()

Delta_r = np.logspace(-20, -23, 50)
resultados["Delta_r"] = Delta_r
resultados["Delta_1"] = vec_delta_1(calcula_Q, r, Delta_r)
resultados["Delta_2"] = vec_delta_2(r, Delta_r)

pd.set_option('display.float_format', lambda x: '%.6e' % x)
display(resultados)

Quando $\Delta r$ é muito pequeno a função $\delta_1$ resulta em um número muito próximo de zero, por conta do fato de que $Q(r+\Delta r) ≈ Q(r)$, logo, para valores muito pequenos de $\Delta r$ temos perda de algarismos significativos por cancelamento, como observado na tabela.

## D.3
À medida que a escala de um sistema é reduzida, a razão entre a área superficial e o volume aumenta inversamente com o tamanho característico (∝ 1/L), intensificando a influência relativa das interações com as fronteiras e dos efeitos viscosos em relação às forças de volume. Isso ocorre porque fenômenos como atrito, tensão superficial e transferência de calor nas paredes escalam com a área, enquanto forças inerciais e efeitos de corpo escalam com o volume. Consequentemente, em micro e nanoescalas, os efeitos de parede e viscosos tornam-se dominantes, frequentemente exigindo modelos físicos diferentes dos usados em macroescala.

# Parte E

In [ ]:


# 1. Parâmetros e Cálculos Físicos
r_ref = 50e-9
mu_ref = 3.5e-3
dp_ref = 5000
L_ref = 10e-6

def calc_Q(r, mu, dp, L):
    return (np.pi * (r**4) * dp) / (8 * mu * L)

Q_ref = calc_Q(r_ref, mu_ref, dp_ref, L_ref)

# Erros de uma variação de 2nm
ea = abs(Q_ref - calc_Q(r_ref + 2e-9, mu_ref, dp_ref, L_ref))
er = ea / Q_ref
ep = er * 100

# Precisão float32 vs float64
r32 = np.float32(r_ref)
erro_f32 = abs(Q_ref - calc_Q(r32, mu_ref, dp_ref, L_ref))

# --- Configuração de Estilo ---
plt.style.use('seaborn-v0_8-darkgrid')
fig = plt.figure(figsize=(20, 16), dpi=100)
fig.patch.set_facecolor('white')

# A. CARTÃO DE DADOS
ax0 = plt.subplot2grid((4, 3), (0, 0), colspan=3)
ax0.axis('off')
ax0.text(0.5, 0.5, f"SÍNTESE TÉCNICA: PROJETO NANOCATETER\n(r: {r_ref*1e9:.0f}nm | DP: {dp_ref}Pa | mu: {mu_ref}Pa.s | Q_ref: {Q_ref:.4e} m3/s)",
         ha='center', va='center', fontsize=22, fontweight='bold',
         bbox=dict(boxstyle='round,pad=1', fc='#2E86C1', ec='none', alpha=0.1))

# B. GRÁFICO DOS TRÊS TIPOS DE ERRO
ax1 = plt.subplot2grid((4, 3), (1, 0))
ax1.bar(['Absoluto', 'Relativo', 'Percentual'], [ea, er, ep], color=['#3498DB', '#1ABC9C', '#E67E22'])
ax1.set_yscale('log')
ax1.set_title('Escala Logarítmica de Erros', fontweight='bold')
ax1.set_ylabel('Valor do Erro')

# C. SENSIBILIDADE Q vs r
ax2 = plt.subplot2grid((4, 3), (1, 1), colspan=2)
r_vals = np.linspace(40e-9, 60e-9, 100)
ax2.plot(r_vals*1e9, calc_Q(r_vals, mu_ref, dp_ref, L_ref), color='#C0392B', lw=3, label='Vazão Q(r)')
ax2.scatter([50], [Q_ref], color='black', zorder=5, label='Ponto Nominal')
ax2.set_title('Sensibilidade Crítica: Vazão vs Raio', fontweight='bold')
ax2.set_xlabel('Raio (nm)')
ax2.set_ylabel('Q (m3/s)')
ax2.legend()

# D. PRECISÃO FLOAT
ax3 = plt.subplot2grid((4, 3), (2, 0))
ax3.bar(['float32', 'float64'], [erro_f32, 1e-36], color=['#9B59B6', '#34495E'])
ax3.set_yscale('log')
ax3.set_title('Resíduo: float32 vs float64', fontweight='bold')
ax3.set_ylabel('Erro de Representação')

# E. EXPLICAÇÃO DOS MÉTODOS
ax4 = plt.subplot2grid((4, 3), (2, 1), colspan=2)
ax4.axis('off')
texto_metodos = ("ANÁLISE DE MÉTODOS E TIPOS DE ERRO:\n"
                 f"- Erro Absoluto (Ea): {ea:.2e} m3/s. Perda real de volume.\n"
                 f"- Erro Percentual (Ep): {ep:.2f}%. Desvio crítico de dosagem.\n"
                 "- Truncamento vs Arredondamento: Em r^4, o truncamento descarta dados de alta magnitude.\n"
                 "- Precisão: float32 introduz ruído em 10^-26; float64 mantém integridade em 10^-32.")
ax4.text(0, 0.5, texto_metodos, fontsize=14, va='center', family='sans-serif',
         bbox=dict(facecolor='#FBFCFC', edgecolor='#D5D8DC', boxstyle='round,pad=0.8'))

# F. CONCLUSÃO FINAL
ax5 = plt.subplot2grid((4, 3), (3, 0), colspan=3)
ax5.axis('off')
conclusao_final = ("CONCLUSÃO TÉCNICA DO RELATÓRIO\n\n"
                   "A análise prova que a precisão em nanoescala é um requisito de segurança médica. "
                   "Devido à sensibilidade do raio a quarta potência, pequenos desvios de arredondamento "
                   "podem levar a falhas de dosagem de até 20%. Recomenda-se o uso estrito de float64.")
ax5.text(0.5, 0.5, conclusao_final, ha='center', va='center', fontsize=16, fontweight='bold', wrap=True,
         color='white', bbox=dict(facecolor='#2E4053', boxstyle='round,pad=1.5'))

plt.tight_layout()
plt.savefig('sintese_final_alinhada.png', dpi=100, bbox_inches='tight')
plt.show()



# 8 - Gif demonstrativo
Resolvemos realizar o gif de duas maneiras: A padrão, utilizando a biblioteca já conhecida, matplotlib e de uma outra maneira, utilizando a biblioteca "Manim", conhecida e amplamente utilizada.

Uma ótima introdução a essa biblioteca é o vídeo do canal "3blue1brown", focado em matemática visualizada: https://www.youtube.com/watch?v=rbu7Zu5X1zI

In [ ]:
# Parâmetros físicos fixos
L_nano = 10e-6
mu_nano = 3.5e-3
dp_nano = 5000
r_nominal = 50e-9

def calc_Q_nano(r_val):
    return (np.pi * (r_val**4) * dp_nano) / (8 * mu_nano * L_nano)

Q_nominal = calc_Q_nano(r_nominal)

# Preparação dos dados para a animação
frames = 60
r_min, r_max = 45e-9, 55e-9
r_vals_anim = np.linspace(r_min, r_max, frames)
Q_vals_anim = [calc_Q_nano(ri) for ri in r_vals_anim]

# Configuração da figura
fig = plt.figure(figsize=(12, 8))
grid = fig.add_gridspec(2, 2)

# Subplot 1: Seção Transversal
ax1 = fig.add_subplot(grid[0, 0])
ax1.set_xlim(-60, 60)
ax1.set_ylim(-60, 60)
ax1.set_aspect('equal')
ax1.set_title("Seção Transversal do Canal", fontweight='bold')
circle = plt.Circle((0, 0), 50, color='#2E86C1', alpha=0.6)
ax1.add_patch(circle)

# Subplot 2: Texto com Dados
ax2 = fig.add_subplot(grid[0, 1])
ax2.axis('off')
text_data = ax2.text(0.1, 0.5, "", fontsize=14, va='center', bbox=dict(boxstyle='round', facecolor='#FEF9E7'))

# Subplot 3: Curva Q vs r Progressiva
ax3 = fig.add_subplot(grid[1, :])
ax3.set_xlim(r_min*1e9, r_max*1e9)
ax3.set_ylim(min(Q_vals_anim), max(Q_vals_anim))
ax3.set_xlabel("Raio (nm)")
ax3.set_ylabel("Vazão Q (m³/s)")
ax3.set_title("Evolução da Vazão Q vs Raio r", fontweight='bold')
line, = ax3.plot([], [], lw=3, color='#E74C3C')
point, = ax3.plot([], [], 'ko')

plt.tight_layout()

def update(i):
    current_r = r_vals_anim[i]
    current_Q = Q_vals_anim[i]
    err_per = abs(current_Q - Q_nominal) / Q_nominal * 100

    # Atualiza Círculo
    circle.set_radius(current_r * 1e9)

    # Atualiza Texto
    info = (f"Raio Atual: {current_r*1e9:.2f} nm\n"
            f"Vazão Q: {current_Q:.2e} m³/s\n"
            f"Erro vs 50nm: {err_per:.2f}%")
    text_data.set_text(info)

    # Atualiza Gráfico
    line.set_data(r_vals_anim[:i+1]*1e9, Q_vals_anim[:i+1])
    point.set_data([current_r*1e9], [current_Q])

    return circle, text_data, line, point

ani = FuncAnimation(fig, update, frames=frames, blit=True)

# Salvar GIF
path_gif = 'animacao_nanocateter.gif'
ani.save(path_gif, writer=PillowWriter(fps=10))
plt.close()

print(f"GIF gerado com sucesso: {path_gif}")

### Visualização da Animação - MatPlotLib

In [ ]:
Image(open('animacao_nanocateter.gif','rb').read())

In [ ]:

class NanoFlowScene(Scene):
    def construct(self):
        # Parâmetros
        r_nominal = 50
        r_range = np.linspace(45, 55, 30)

        def get_flow(r_nm):
            return (r_nm / 50)**4

        # Títulos - Fixado no topo
        title = Text("Dinâmica do Nanocateter", font_size=36).to_edge(UP, buff=0.3)
        self.add(title)

        # 1. Seção Transversal (Círculo) - Reposicionado para evitar o título
        circle = Circle(radius=r_nominal/20, color=BLUE, fill_opacity=0.5)
        circle_label = Text("Seção Transversal", font_size=24)
        circle_group = VGroup(circle, circle_label).arrange(DOWN, buff=0.3)
        circle_group.to_corner(UL, buff=1.5)

        # 2. Eixos para o Gráfico
        axes = Axes(
            x_range=[44, 56, 2],
            y_range=[0.6, 1.6, 0.2],
            x_length=5,
            y_length=3.5,
            axis_config={"include_tip": True}
        ).to_edge(RIGHT, buff=0.5).shift(DOWN*0.5)

        x_lab = Text("r (nm)", font_size=20).next_to(axes.x_axis, RIGHT)
        y_lab = Text("Q / Q_50", font_size=20).next_to(axes.y_axis, UP)

        # Elementos dinâmicos
        first_point = axes.c2p(r_range[0], get_flow(r_range[0]))
        dot = Dot(point=first_point, color=RED)
        path = VMobject(color=YELLOW)
        path.set_points_as_corners([first_point, first_point])

        # Textos de status
        r_val_text = Text(f"Raio: {r_range[0]:.2f} nm", font_size=24)
        err_val_text = Text(f"Erro: 0.00 %", font_size=24)

        status_display = VGroup(r_val_text, err_val_text).arrange(RIGHT, buff=1).to_edge(DOWN, buff=0.8).shift(DOWN*0.5)

        self.add(circle_group, axes, x_lab, y_lab, status_display, dot, path)

        # Animação
        for i, r_val in enumerate(r_range):
            new_radius = r_val / 20
            new_q = get_flow(r_val)
            new_point = axes.c2p(r_val, new_q)
            current_err = abs(new_q - 1) * 100

            new_r_content = Text(f"Raio: {r_val:.2f} nm", font_size=24).move_to(r_val_text)
            new_err_content = Text(f"Erro: {current_err:.2f} %", font_size=24).move_to(err_val_text)

            self.play(
                circle.animate.set_radius(new_radius),
                dot.animate.move_to(new_point),
                r_val_text.animate.become(new_r_content),
                err_val_text.animate.become(new_err_content),
                run_time=0.05,
                rate_func=linear
            )
            path.add_points_as_corners([new_point])

        self.wait(2)

# ===== FUNÇÃO PARA LIMPAR O CACHE =====
def limpar_manim_cache():
    """Remove diretórios de cache do Manim"""
    pastas = [
        "./partial_movie_files",
        "./media",
        "./video",
        "./gif",
        "./pngs"
    ]
    for pasta in pastas:
        if os.path.exists(pasta):
            try:
                shutil.rmtree(pasta)
                print(f"✅ Removido: {pasta}")
            except Exception as e:
                print(f"⚠️ Erro ao remover {pasta}: {e}")

# ===== EXECUÇÃO =====
if __name__ == "__main__":
    # Primeiro, limpa o cache
    limpar_manim_cache()

    # Configurações
    config.frame_rate = 45  # Mudar para 45 fps
    config.format = "gif"
    config.video_dir = "./"
    config.output_file = "NanoFlowScene"

    try:
        scene = NanoFlowScene()
        scene.render()
        print("✅ Animação criada com sucesso!")
    except Exception as e:
        print(f"❌ Erro: {e}")
        print("Tentando novamente com limpeza completa...")
        limpar_manim_cache()
        scene = NanoFlowScene()
        scene.render()

In [ ]:
from IPython.display import Image, display
import os

gif_path = "NanoFlowScene.gif"
fallback_path = "NanoFlowScene.mp4.gif"

if os.path.exists(gif_path):
    display(Image(gif_path))
elif os.path.exists(fallback_path):
    display(Image(fallback_path))
else:
    print("Arquivo de animação não encontrado. Por favor, execute a célula anterior primeiro.")

# Parte E - Síntese vísual com GIF (EXTRA)

In [ ]:

# 1. Configurações Físicas e Funções de Suporte
r_nom = 50e-9
dp = 5000
mu = 3.5e-3
L = 10e-6

def calc_Q(r): return (np.pi * (r**4) * dp) / (8 * mu * L)
Q_nom = calc_Q(r_nom)

def truncar(x, n):
    fator = 10**n
    return math.trunc(x * fator) / fator

# Preparação da Animação (Variação de +-5nm)
frames = 40
r_vals = np.linspace(45e-9, 55e-9, frames)

# 2. Configuração do Dashboard Animado
plt.style.use('seaborn-v0_8-darkgrid')
fig = plt.figure(figsize=(20, 16))
grid = fig.add_gridspec(4, 3, height_ratios=[1, 1, 1, 0.5], hspace=0.4, wspace=0.3)

# A. Subplot: Seção Transversal Dinâmica
ax_circ = fig.add_subplot(grid[0, 0])
ax_circ.set_aspect('equal')
circle = plt.Circle((0, 0), 50, color='#2E86C1', alpha=0.6)
ax_circ.add_patch(circle)
ax_circ.set_xlim(-60, 60); ax_circ.set_ylim(-60, 60)
ax_circ.set_title("Seção Transversal (nm)", fontweight='bold')

# B. Subplot: Sensibilidade Q vs r
ax_sens = fig.add_subplot(grid[0, 1:])
r_plot = np.linspace(40e-9, 60e-9, 100)
ax_sens.plot(r_plot*1e9, [calc_Q(ri) for ri in r_plot], color='gray', alpha=0.3, ls='--')
line_sens, = ax_sens.plot([], [], color='#C0392B', lw=3, label='Trajetória Q(r)')
dot_sens, = ax_sens.plot([], [], 'ko', markersize=8)
ax_sens.set_title("Sensibilidade Dinâmica: Vazão vs Raio", fontweight='bold')
ax_sens.set_ylabel("Q (m³/s)")

# C. Subplot: Comparação de Erros em Tempo Real
ax_err = fig.add_subplot(grid[1, 0])
bar_err = ax_err.bar(['Abs', 'Rel', 'Perc'], [1e-32, 1e-32, 1e-32], color=['#3498DB', '#1ABC9C', '#E67E22'])
ax_err.set_yscale('log')
ax_err.set_ylim(1e-33, 1e2)
ax_err.set_title("Erros da Dosagem Atual", fontweight='bold')

# D. Subplot: Histograma de Pressão (Dados da Parte B)
temp_data = np.array([121.7, 120.4, 122.1, 119.8, 121.2, 120.9, 122.4, 121.0, 120.2, 121.5])
ax_hist = fig.add_subplot(grid[1, 1])
ax_hist.hist(temp_data, bins=5, color='#AED6F1', edgecolor='white')
ax_hist.set_title("Distribuição de Pressão (Parte B)", fontweight='bold')
ax_hist.set_xlabel("P (Pa)")

# E. Subplot: Eficácia Arredondamento vs Truncamento (Dados da Parte A)
ax_met = fig.add_subplot(grid[1, 2])
n_vals_plot = np.arange(5)
ea_arr = [abs(np.pi - round(np.pi, n)) for n in n_vals_plot]
ea_tru = [abs(np.pi - truncar(np.pi, n)) for n in n_vals_plot]
ax_met.plot(n_vals_plot, ea_arr, 'o-', label='Arred.')
ax_met.plot(n_vals_plot, ea_tru, 's--', label='Trunc.')
ax_met.set_yscale('log')
ax_met.set_title("Precisão por Método (Parte A)", fontweight='bold')
ax_met.legend(fontsize=8)

# F. Bloco de Texto de Status
ax_txt = fig.add_subplot(grid[2, :])
ax_txt.axis('off')
status_txt = ax_txt.text(0.5, 0.5, "", ha='center', va='center', fontsize=15,
                         bbox=dict(boxstyle='round,pad=1', fc='#FBFCFC', ec='#D5D8DC'))

# G. Conclusão Fixa Integrada
ax_conc = fig.add_subplot(grid[3, :])
ax_conc.axis('off')
ax_conc.text(0.5, 0.5, "SÍNTESE FINAL: A sensibilidade r^4 exige float64 e arredondamento simétrico.\n"
             "Pequenos desvios mecânicos invalidam a dosagem e representam risco clínico.",
             ha='center', va='center', fontsize=16, fontweight='bold', color='white',
             bbox=dict(facecolor='#2E4053', boxstyle='round,pad=1'))

def update(i):
    curr_r = r_vals[i]
    curr_Q = calc_Q(curr_r)
    ea = abs(Q_nom - curr_Q)
    er = ea / Q_nom if Q_nom != 0 else 0
    ep = er * 100

    # Atualiza Círculo
    circle.set_radius(curr_r * 1e9)

    # Atualiza Gráfico Sensibilidade
    line_sens.set_data(r_vals[:i+1]*1e9, [calc_Q(ri) for ri in r_vals[:i+1]])
    dot_sens.set_data([curr_r*1e9], [curr_Q])

    # Atualiza Barras de Erro
    for rect, val in zip(bar_err, [ea, er, ep]):
        rect.set_height(max(val, 1e-32))

    # Atualiza Texto
    info = (f"ESTADO DINÂMICO DO NANOCATETER\n"
            f"Raio: {curr_r*1e9:.2f} nm  |  Vazão: {curr_Q:.2e} m³/s\n"
            f"Erro Relativo: {er:.4f}  |  Desvio Percentual: {ep:.2f}%")
    status_txt.set_text(info)

    return [circle, line_sens, dot_sens, status_txt] + list(bar_err)

ani = FuncAnimation(fig, update, frames=frames, blit=True)
ani.save('dashboard_nanocateter_completo.gif', writer=PillowWriter(fps=8))
plt.close()

display(Image(filename='dashboard_nanocateter_completo.gif'))